# 02 — Feature engineering

### Objetivo
Crear features predictivas a partir de `df_semanal.csv`: lags, rolling stats, calendario, OHE.

### Cuidado con leakage
Toda feature que use el pasado se calcula con `shift(1)` para que la fila de la semana `t` solo vea info hasta `t-1`.

In [10]:
# Carga del dataset semanal
import pandas as pd
import numpy as np
from pathlib import Path
import holidays as hl
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path("/Users/guillermo/Downloads/TA_IA_Aplicada/Data")
SRC = DATA_DIR / "df_semanal.csv"
DST = DATA_DIR / "df_features.csv"

df = pd.read_csv(SRC)
print(f"Shape: {df.shape}")

Shape: (12728, 7)


### Features temporales ciclicas

In [11]:
# Semana ISO + encoding ciclico (sin/cos) para semana_del_anio y mes
df["fecha_inicio_semana"] = pd.to_datetime(
    df["iso_year"].astype(str) + "-W" + df["iso_week"].astype(str).str.zfill(2) + "-1",
    format="%G-W%V-%u",
)
df["mes"] = df["fecha_inicio_semana"].dt.month
df["año"] = df["iso_year"]  # usa anio ISO, no calendario (evita caso W01 -> 30-dic-anterior)

df["semana_del_año_sin"] = np.sin(2 * np.pi * df["iso_week"] / 52)
df["semana_del_año_cos"] = np.cos(2 * np.pi * df["iso_week"] / 52)
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

### Festivos en Peru

In [12]:
# Semana con festivo en Peru (algun dia de la semana cae en fecha festiva)
pe_holidays = hl.country_holidays("PE", years=range(2025, 2027))
fechas_festivas = set(pe_holidays.keys())

def semana_tiene_festivo(fecha_inicio):
    for d in range(7):
        f = fecha_inicio + pd.Timedelta(days=d)
        if f.date() in fechas_festivas:
            return 1
    return 0

df["semana_con_festivo"] = df["fecha_inicio_semana"].apply(semana_tiene_festivo)
print(f"Semanas con festivo: {df['semana_con_festivo'].sum()} de {df['año_semana'].nunique()}")

Semanas con festivo: 2752 de 74


### Indice temporal continuo

In [13]:
# Semana global: indice entero monotono para usar como feature
df["semana_del_año"] = df["iso_week"]
df["semana_global"] = (df["iso_year"] - df["iso_year"].min()) * 52 + df["iso_week"]

### Lags del target

In [14]:
# Lags por grupo (distrito, turno): la fila t ve count_robos de t-1, t-2, ..., t-52
df = df.sort_values(["distrito_hecho", "turno_hecho", "semana_global"]).reset_index(drop=True)
grp = df.groupby(["distrito_hecho", "turno_hecho"])["count_robos"]

df["count_lag_1w"]  = grp.shift(1)
df["count_lag_2w"]  = grp.shift(2)
df["count_lag_4w"]  = grp.shift(4)
df["count_lag_8w"]  = grp.shift(8)
df["count_lag_52w"] = grp.shift(52)

### Rolling statistics

In [15]:
# Media y std moviles con ventana 4, 8, 12 semanas (shift(1) para no incluir la fila actual)
df["rolling_mean_4w"]  = grp.shift(1).rolling(window=4,  min_periods=1).mean()
df["rolling_mean_8w"]  = grp.shift(1).rolling(window=8,  min_periods=1).mean()
df["rolling_mean_12w"] = grp.shift(1).rolling(window=12, min_periods=1).mean()
df["rolling_std_4w"]   = grp.shift(1).rolling(window=4,  min_periods=1).std()
df["rolling_std_8w"]   = grp.shift(1).rolling(window=8,  min_periods=1).std()

### Diferencias y media historica estacional

In [16]:
# Media historica por (distrito, turno, semana_del_anio)
df["media_historica"] = (
    df.groupby(["distrito_hecho", "turno_hecho", "iso_week"])["count_robos"]
      .transform(lambda s: s.shift(1).expanding().mean())
)

### Encoding categorico

In [17]:
# OHE para turno y subtipo; LabelEncoder para distrito
df = pd.get_dummies(df, columns=["turno_hecho"], prefix="turno")
df = pd.get_dummies(df, columns=["subtipo_hecho"], prefix="subtipo")

le_distrito = LabelEncoder()
df["distrito_id"] = le_distrito.fit_transform(df["distrito_hecho"])
print(f"Distritos: {df['distrito_hecho'].nunique()}")

Distritos: 43


### Guardado y limpieza de nulos

In [18]:
# Seleccion de columnas y guardado; los nulos en lags son semanas iniciales sin historia
cols_to_save = [
    "distrito_hecho", "año_semana", "semana_global",
    "iso_year", "iso_week", "mes", "año", "semana_del_año",
    "distrito_id",
    "count_robos",
    "semana_del_año_sin", "semana_del_año_cos", "mes_sin", "mes_cos",
    "semana_con_festivo",
    "count_lag_1w", "count_lag_2w", "count_lag_4w", "count_lag_8w", "count_lag_52w",
    "rolling_mean_4w", "rolling_mean_8w", "rolling_mean_12w",
    "rolling_std_4w", "rolling_std_8w",
    "media_historica",
] + [c for c in df.columns if c.startswith("turno_")] + [c for c in df.columns if c.startswith("subtipo_")]

df_out = df[cols_to_save].copy()
nulos = df_out.isnull().sum()
if nulos.sum() > 0:
    print(f"Columnas con nulos (rellenar con 0): {(nulos > 0).sum()}")
    df_out = df_out.fillna(0)
df_out.to_csv(DST, index=False)
print(f"Guardado: {DST} | Shape: {df_out.shape} | Nulos: {df_out.isnull().sum().sum()}")

Columnas con nulos (rellenar con 0): 11
Guardado: /Users/guillermo/Downloads/TA_IA_Aplicada/Data/df_features.csv | Shape: (12728, 31) | Nulos: 0


### Output
- `Data/df_features.csv` (12,728 filas x 33 columnas) listo para entrenar.
- 33 features: 5 calendario + 5 lags + 5 rolling + 2 diff/ratio + 1 historica + 5 OHE turno + 1 OHE subtipo + 5 metadata + 1 distrito_id.